# 02 - Preprocesamiento, split temporal y features

Split temporal (out-of-time), encoding de categóricas, log del monto, manejo de faltantes. Todo ajustado SOLO en train.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_raw
from src.features import temporal_split
from src import utils
from src.utils import PALETA, PALETA_CLASES


utils.set_style()
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Columnas de trabajo (una sola definición, como en el EDA)
NUM_COLS = ["b", "c", "d", "e", "f", "h", "k", "l", "m", "monto", "score"]
CAT_COLS = ["a", "n", "o", "p", "g"]
TARGET = "fraude"

# Carga
df = load_raw("../data/dataset.csv")
print("Dataset cargado:", df.shape)

Dataset cargado: (150000, 19)


### Split temporal (out-of-time)

In [2]:
train, val, test = temporal_split(df, train_frac=0.7, val_frac=0.15)

# Verificar el split: rangos de fecha y tasa de fraude en cada partición
for nombre, parte in [("train", train), ("val", val), ("test", test)]:
    print(f"{nombre}: {len(parte):>6} filas | "
          f"{parte['fecha'].min().date()} → {parte['fecha'].max().date()} | "
          f"fraude {parte['fraude'].mean():.2%}")

train: 105000 filas | 2020-03-08 → 2020-04-10 | fraude 5.18%
val:  22500 filas | 2020-04-10 → 2020-04-16 | fraude 4.53%
test:  22500 filas | 2020-04-16 → 2020-04-21 | fraude 4.60%


Se ordena por fecha y se parte en el tiempo — **train (70%), val (15%),
test (15%)** — para simular producción: se entrena con el pasado y se evalúa
con el futuro.

| Partición | Filas   | Rango           | Tasa fraude |
|-----------|---------|-----------------|-------------|
| train     | 105.000 | 08/03 → 10/04   | 5,18%       |
| val       | 22.500  | 10/04 → 16/04   | 4,53%       |
| test      | 22.500  | 16/04 → 21/04   | 4,60%       |

**Por qué 70/15/15.** La validación se usa para calibrar y elegir el umbral de
decisión, y esa elección debe transferirse al test (el futuro). Con este
reparto, las tasas de fraude de val (4,53%) y test (4,60%) son casi idénticas,
por lo que la validación es un ensayo fiel del test. Un reparto alternativo
(66/14/20) dejaba la val más parecida al train que al test, menos
representativa para ajustar el umbral. Se mantiene el enfoque de validación
cronológica de la literatura de referencia (el porcentaje exacto no altera el
método).


## Limpieza

In [3]:
from src.data import clean

train = clean(train)
val = clean(val)
test = clean(test)

# Definir los grupos de columnas para el preprocesamiento posterior
NUM_FEATURES = ["b", "c", "d", "e", "f", "h", "l", "m", "monto", "score"]
CAT_FEATURES = ["a", "g", "j", "n", "o", "p"]

print("Features numéricas:", len(NUM_FEATURES))
print("Features categóricas:", len(CAT_FEATURES))
print("Columnas de train:", train.shape[1])

Features numéricas: 10
Features categóricas: 6
Columnas de train: 18


## Transformaciones

In [4]:
from src.features import build_preprocessor, NUM_FEATURES, CAT_LOW, CAT_HIGH

FEATURES = NUM_FEATURES + CAT_LOW + CAT_HIGH

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

pre = build_preprocessor()
X_train_p = pre.fit_transform(X_train, y_train)   # ajusta SOLO en train
X_val_p   = pre.transform(X_val)                  # aplica lo aprendido
X_test_p  = pre.transform(X_test)

print("Shape tras preprocesar:", X_train_p.shape)

Shape tras preprocesar: (105000, 27)


odas las transformaciones se ajustan **solo con el train** (`fit_transform` en
train, `transform` en val/test), para evitar fugas de información hacia la
evaluación.

Cada tipo de columna recibe un tratamiento distinto (`ColumnTransformer`):

- **Numéricas** (`b, c, d, e, f, h, l, m, monto, score`): imputación de
  faltantes por **mediana** (robusta a los outliers detectados en el EDA).
- **Categóricas de baja cardinalidad** (`a, g, n, o, p`): imputación por moda +
  **one-hot encoding**, agrupando las categorías poco frecuentes (p. ej. países
  raros) y tolerando categorías nuevas en val/test.
- **Categórica de alta cardinalidad** (`j`, 8.324 valores): **target encoding**
  con validación cruzada y suavizado, que reemplaza cada categoría por su tasa
  de fraude sin sobreajustar las categorías raras.


Resultado: las 16 features de entrada se expanden a **27 columnas** (por el
one-hot), todas numéricas y sin faltantes, listas para el modelo:
`X_train_p.shape = (105.000, 27)`.